# 01 — Data Cleaning
Raw CSV → exclusion cascade → cleaned, reverse-scored analytic dataset.

Set `RAW_CSV_PATH` below to your raw survey export. The individual-level survey
export is not included in this repository (see `data/README.md`); supply your
own export with the same 33-column structure to run this notebook.

In [ ]:
import pandas as pd
import numpy as np

RAW_CSV_PATH = "../data/raw_survey_export.csv"  # <-- CHANGE THIS

df = pd.read_csv(RAW_CSV_PATH)
print("Raw N =", len(df))

before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"After removing exact duplicate submissions: {len(df)} (dropped {before - len(df)})")

In [ ]:
df1 = df[df.iloc[:, 2] == "Yes, I consent to participate."].copy()
print(f"After consent filter: {len(df1)} (dropped {len(df) - len(df1)})")

ac1_pass = df1.iloc[:, 11] == "Disagree"
ac2_pass = df1.iloc[:, 27] == "Almost Always"
print("Failed attention check 1:", (~ac1_pass).sum())
print("Failed attention check 2:", (~ac2_pass).sum())
df2 = df1[ac1_pass & ac2_pass].copy()
print(f"After attention-check filter: {len(df2)} (dropped {len(df1) - len(df2)})")

In [ ]:
AGREE_MAP = {"Strongly Disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly Agree": 5}
FREQ_MAP  = {"Never": 1, "Rarely": 2, "Sometimes": 3, "Often": 4, "Almost Always": 5}
mmr_cols = list(range(6, 11)); aiu_cols = list(range(12, 17)); ail_cols = list(range(17, 22))
co_cols  = list(range(22, 27)); cte_cols = list(range(28, 33))
all_cols = mmr_cols + aiu_cols + ail_cols + co_cols + cte_cols
agree_idx = set(mmr_cols) | set(ail_cols)

numeric = pd.DataFrame(index=df2.index)
for i in all_cols:
    mapping = AGREE_MAP if i in agree_idx else FREQ_MAP
    numeric[f"c{i}"] = df2.iloc[:, i].map(mapping)

n_nan = numeric.isna().sum().sum()
print("Unmapped/missing values:", n_nan)
if n_nan > 0:
    print("Dropping incomplete rows (likely abandoned submissions).")
    numeric = numeric.dropna()
    df2 = df2.loc[numeric.index]
    print(f"N after dropping incomplete rows: {len(numeric)}")

In [ ]:
row_variance = numeric.var(axis=1)
straightliners = row_variance < 0.01
print("Straightliners detected:", straightliners.sum())
numeric = numeric[~straightliners].copy()
df2_final = df2.loc[numeric.index]
print(f"Final analytic N = {len(numeric)}")

## Reverse-score the five reverse-worded items

One item per construct is reverse-worded (c10, c12, c21, c26, c32). Reverse-score
them (6 minus the raw value on this 1–5 scale) **before** any reliability or
item-quality diagnostics.

In [ ]:
reverse_items = {"MMR": "c10", "AIU": "c12", "AIL": "c21", "CO": "c26", "CTE": "c32"}
df_reversed = numeric.copy()
for construct, col in reverse_items.items():
    df_reversed[col] = 6 - df_reversed[col]

meta = pd.DataFrame(index=df2_final.index)
meta["level_of_study"] = df2_final.iloc[:, 3].values
meta["field_of_study"] = df2_final.iloc[:, 4].values
meta["usage_time"] = df2_final.iloc[:, 5].values
meta["timestamp"] = df2_final.iloc[:, 0].values

df_reversed.to_csv("items_numeric_reversed.csv")
meta.to_csv("meta.csv")
print("Saved items_numeric_reversed.csv:", df_reversed.shape)
print("Saved meta.csv:", meta.shape)

In [ ]:
print("Level of study:\n", meta.level_of_study.value_counts(), "\n")
print("Field of study:\n", meta.field_of_study.value_counts(), "\n")
print("Daily AI usage:\n", meta.usage_time.value_counts())

**Next notebook:** `02_measurement_model.ipynb`